# Phase 2 · The state scorecard

> **A screening tool, not investment advice.**

**The question for this phase:** if I had to say where in Nigeria to look first for a specialist
care centre, which states come up, and how sure can I be?

The scorecard combines three things for each state, called **pillars**:

| Pillar | The plain question | Higher score means |
|---|---|---|
| **Need** | How many people might need specialist care? | more need |
| **Supply gap** | How little hospital capacity is there for them? | bigger gap |
| **Ability to pay** | Can people (or their insurance) pay for care? | more able to pay |

Every number below is printed by code from `data/processed/`. None are typed in.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import plotly.io as pio

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT / "pipeline"), str(ROOT / "src")]
import scoring as S, charts

pio.renderers.default = "plotly_mimetype"
pd.set_option("display.max_rows", 60, "display.width", 160)

P = ROOT / "data" / "processed"
panel = pd.read_csv(P / "state_panel.csv")
sc = pd.read_csv(P / "scorecard.csv")
st = pd.read_csv(P / "scorecard_stress.csv")
sens = pd.read_csv(P / "scorecard_sensitivity_need.csv")
print(S.LABEL)

A screening tool, not investment advice.


## 1. What goes into each pillar

In [2]:
from_dict = pd.read_csv(P / "state_panel_dictionary.csv").set_index("column")
rows = []
for pillar, inputs in S.PILLARS.items():
    for col, direction in inputs:
        base = col.replace("_avg_pct", "_women_pct")  # averages are built from the women's and men's columns
        desc = from_dict.description.get(col, from_dict.description.get(base, ""))
        rows.append({"pillar": S.PILLAR_NAMES[pillar], "input": col,
                     "higher value means": "higher pillar score" if direction > 0 else "LOWER pillar score (flipped)",
                     "what it is": desc + (" (women and men averaged)" if "_avg_" in col else "")})
pd.DataFrame(rows)

,pillar,input,higher value means,what it is
0,Need,population_total,higher pillar score,"Modelled population, 2025 estimate, rounded to..."
1,Supply gap,fac_sec_tert_per_100k,LOWER pillar score (flipped),"Secondary + tertiary facilities per 100,000 pe..."
2,Ability to pay,insurance_any_avg_pct,higher pillar score,Women 15-49 with any health insurance (women a...
3,Ability to pay,wealth_top2_pct,higher pillar score,Household population in top two wealth quintiles


**Two honest limits on these inputs**
- **Need** should use population aged 40+, the age group where kidney failure is most common.
  The population file has no age groups, so total population stands in for now.
- The blood pressure and blood sugar figures are **self-reported** ("ever told by a health worker"),
  for ages 15–49. You can only be told if you were tested, so they rise where testing is common.
  So they are **not** in the main Need score (updated after review): Need = population. Section 7
  shows what changes if they are added.

## 2. How the scores are made: percentile ranks

Each input is turned into a **percentile rank** from 0 to 100: 0 is the lowest of the 37 states,
100 the highest. Ranks are used instead of raw values so one extreme state can't squash everyone
else into a narrow band. Here's the effect on the wealth input:

In [3]:
w = panel[["state", "wealth_top2_pct"]].assign(rank=S.percentile_rank(panel.wealth_top2_pct).round(1))
print("Raw values: the gap between the top state and the next one is "
      f"{w.wealth_top2_pct.nlargest(2).diff().abs().iloc[-1]:.1f} points; on the rank scale every step is "
      f"{100 / (len(w) - 1):.1f} points.")
w.sort_values("rank", ascending=False).head(6)

Raw values: the gap between the top state and the next one is 10.4 points; on the rank scale every step is 2.8 points.


,state,wealth_top2_pct,rank
24,Lagos,97.5,100.0
11,Edo,87.1,97.2
3,Anambra,84.6,94.4
14,Federal Capital Territory,82.1,91.7
29,Osun,80.4,88.9
27,Ogun,79.1,86.1


Then:
1. Where less means more (fewer hospitals = bigger supply gap), the rank is **flipped**.
2. A pillar is the average of its inputs' ranks, **ranked again** from 0 to 100.
3. The **total** is the average of the three pillars, each weighted 1/3.

## 3. The full scorecard (equal weights)

In [4]:
sc[["rank", "state", "zone", "score_need", "score_supply_gap", "score_ability_to_pay", "score_total"]]

,rank,state,zone,score_need,score_supply_gap,score_ability_to_pay,score_total
0,1,Kano,North West,100.0,90.3,63.9,84.7
1,2,Ogun,South West,97.2,45.8,84.7,75.9
2,3,Kaduna,North West,91.7,70.8,61.1,74.5
3,3,Katsina,North West,88.9,98.6,36.1,74.5
4,5,Lagos,South West,94.4,11.1,100.0,68.5
5,6,Edo,South South,66.7,41.7,95.8,68.1
6,7,Jigawa,North West,75.0,98.6,27.8,67.1
7,8,Bauchi,North East,80.6,90.3,25.0,65.3
8,9,Rivers,South South,69.4,52.8,72.2,64.8
9,10,Delta,South South,77.8,38.9,75.0,63.9


## 4. Both ends of the ranking

The total alone hides *why* a state ranks where it does, so the chart shows each pillar.

In [5]:
charts.scorecard_ends(sc).show()

In [6]:
top, bottom = sc.head(5), sc.tail(5)
def pillar_extremes(d):
    return {S.PILLAR_NAMES[p]: round(d[f"score_{p}"].mean()) for p in S.PILLARS}
print("Average pillar scores, top 5:   ", pillar_extremes(top))
print("Average pillar scores, bottom 5:", pillar_extremes(bottom))

Average pillar scores, top 5:    {'Need': 94, 'Supply gap': 63, 'Ability to pay': 69}
Average pillar scores, bottom 5: {'Need': 16, 'Supply gap': 25, 'Ability to pay': 32}


## 5. How sure can I be? The weight stress-test

Equal weights are a choice. I re-ran the total for every combination of weights from 0.2 to 0.5
per pillar (in steps of 0.05, rescaled to add up to 1) and counted how often each state lands in
the top 10. A state that stays near the top under most weightings is a **robust** finding.

In [7]:
charts.stress_chart(st).show()

In [8]:
share = f"share_in_top{S.TOP_N}_pct"
always = st[st[share] == 100].state.tolist()
print(f"{len(S.weight_grid())} weightings tested.")
print(f"In the top {S.TOP_N} under every weighting: {', '.join(always)}")
print(f"Top {S.TOP_N} under some weightings only: "
      f"{', '.join(st[(st[share] > 0) & (st[share] < 80)].state)}")

319 weightings tested.
In the top 10 under every weighting: Kano, Ogun, Kaduna
Top 10 under some weightings only: Jigawa, Bauchi, Osun, Delta, Benue, Ondo, Oyo, Yobe, Niger


### Try your own weights
Change the numbers below and re-run the cell. They don't need to add up to 1; they're rescaled.

In [9]:
my_weights = {"need": 1, "supply_gap": 1, "ability_to_pay": 1}   # <- edit me

mine = S.build_scorecard(panel, weights=my_weights)
print("Weights used:", {k: round(v, 2) for k, v in S.normalise(my_weights).items()})
mine[["rank", "state", "score_need", "score_supply_gap", "score_ability_to_pay", "score_total"]].head(10)

Weights used: {'need': 0.33, 'supply_gap': 0.33, 'ability_to_pay': 0.33}


,rank,state,score_need,score_supply_gap,score_ability_to_pay,score_total
0,1,Kano,100.0,90.3,63.9,84.7
1,2,Ogun,97.2,45.8,84.7,75.9
2,3,Kaduna,91.7,70.8,61.1,74.5
3,3,Katsina,88.9,98.6,36.1,74.5
4,5,Lagos,94.4,11.1,100.0,68.5
5,6,Edo,66.7,41.7,95.8,68.1
6,7,Jigawa,75.0,98.6,27.8,67.1
7,8,Bauchi,80.6,90.3,25.0,65.3
8,9,Rivers,69.4,52.8,72.2,64.8
9,10,Delta,77.8,38.9,75.0,63.9


## 6. Testing the tension: does the need sit where people can't pay?

The brief asks me to **test, not assume**, that places with the greatest need may have the least
ability to pay. I checked two versions of "need".

In [10]:
charts.need_vs_pay(sc).show()

In [11]:
charts.gap_vs_pay(sc).show()

In [12]:
r_need = charts.spearman(sc, "score_need", "score_ability_to_pay")
r_gap = charts.spearman(sc, "score_supply_gap", "score_ability_to_pay")
alt = S.pillar_scores(panel, S.PILLARS_NEED_WITH_DIAGNOSES)
r_need_dx = charts.spearman(alt, "score_need", "score_ability_to_pay")
quad = sc[(sc.score_supply_gap > 50) & (sc.score_ability_to_pay < 50)]
print(f"Need (population, main score) vs ability to pay:      rank correlation {r_need:+.2f}")
print(f"Need (with self-reported diagnoses) vs ability to pay: rank correlation {r_need_dx:+.2f}")
print(f"Supply gap vs ability to pay:             rank correlation {r_gap:+.2f}")
print(f"\n{len(quad)} states have an above-middle supply gap AND below-middle ability to pay:")
print(", ".join(quad.sort_values("score_supply_gap", ascending=False).state))

Need (population, main score) vs ability to pay:      rank correlation +0.22
Need (with self-reported diagnoses) vs ability to pay: rank correlation +0.53
Supply gap vs ability to pay:             rank correlation -0.52

13 states have an above-middle supply gap AND below-middle ability to pay:
Katsina, Jigawa, Yobe, Bauchi, Adamawa, Sokoto, Taraba, Zamfara, Borno, Niger, Gombe, Benue, Kebbi


**What I read from this**

- **The supply-gap view supports the tension.** States with the fewest hospitals per person tend
  to have the least insurance and the fewest better-off households (the negative correlation above).
- **The need view depends on how need is measured.** With population alone (the main score), compare
  the first correlation above with the second: adding self-reported diagnoses pushes need towards
  ability to pay, because "ever told by a health worker" rises where people get tested, and testing is
  more common in better-off states. That is why the diagnoses stay out of the main score.
- Age-group population (40+), if it is ever published by state, would give a fairer need measure.

## 7. Sensitivity: Need with the self-reported figures added

In [13]:
overlap = len(set(sc.head(10).state) & set(sens.nsmallest(10, "rank_need_with_diagnoses").state))
print(f"{overlap} of the top 10 are the same either way.")
movers = sens.reindex(sens.rank_change.abs().sort_values(ascending=False).index).head(8)
movers[["state", "rank", "rank_need_with_diagnoses", "rank_change"]].rename(
    columns={"rank": "rank (main: population)", "rank_change": "places moved when diagnoses are added (+ = down)"})

8 of the top 10 are the same either way.


,state,rank (main: population),rank_need_with_diagnoses,places moved when diagnoses are added (+ = down)
14,Niger,15,27,12
26,Bayelsa,27,15,-12
3,Katsina,3,14,11
6,Jigawa,7,16,9
20,Nasarawa,18,9,-9
28,Kogi,29,20,-9
12,Ondo,13,4,-9
13,Oyo,14,22,8


## 8. Choosing Halima's state (for you to confirm)

Halima is an **illustrative** character: 52, a trader whose kidneys have failed. The brief says her
state is chosen **from the data**: high need and little capacity. So I rank states on Need and
Supply gap only, leaving out ability to pay (her family's struggle to pay is part of the story, not
a reason to pick or skip a state). I show both versions of Need.

In [14]:
def halima_candidates(scores):
    return (scores.assign(need_and_gap=(scores.score_need + scores.score_supply_gap) / 2)
                  .sort_values("need_and_gap", ascending=False)
                  [["state", "score_need", "score_supply_gap", "score_ability_to_pay", "need_and_gap"]].head(6))

a = halima_candidates(S.pillar_scores(panel)).reset_index(drop=True)
b = halima_candidates(S.pillar_scores(panel, S.PILLARS_NEED_WITH_DIAGNOSES)).reset_index(drop=True)
both = sorted(set(a.state.head(5)) & set(b.state.head(5)), key=lambda s: a.state.tolist().index(s) if s in a.state.tolist() else 99)
print("In the top 5 under BOTH versions of Need:", ", ".join(both))
print("\nNeed = population (main):"); print(a.round(1).to_string(index=False))
print("\nNeed with self-reported diagnoses (check):"); print(b.round(1).to_string(index=False))

In the top 5 under BOTH versions of Need: Kano, Bauchi, Kaduna

Need = population (main):
  state  score_need  score_supply_gap  score_ability_to_pay  need_and_gap
   Kano       100.0              90.3                  63.9          95.2
Katsina        88.9              98.6                  36.1          93.8
 Jigawa        75.0              98.6                  27.8          86.8
 Bauchi        80.6              90.3                  25.0          85.4
 Kaduna        91.7              70.8                  61.1          81.2
  Benue        86.1              59.7                  41.7          72.9

Need with self-reported diagnoses (check):
  state  score_need  score_supply_gap  score_ability_to_pay  need_and_gap
   Kano        83.3              90.3                  63.9          86.8
 Bauchi        76.4              90.3                  25.0          83.4
 Kaduna        72.2              70.8                  61.1          71.5
 Rivers        88.9              52.8               

## 9. Checks

In [15]:
import validate
errors = validate.check_scorecard()
print("Scorecard checks passed: 37 rows, scores 0-100, label on every row." if not errors else errors)

Scorecard checks passed: 37 rows, scores 0-100, label on every row.